# Job ETL da silver

Neste notebook, é aplicado o Job ETL. Ele é um processo que extrai, transforma e carrega dados de diferentes fontes para um destino central. No caso desse projeto, a fonte será extraida da camada silver para a gold pelo arquivo Complete_Pokedex-Tratada.csv e o resultado será armazenado em outro csv e utilizado na camada gold.

### Frameworks utilizados

In [3]:
import pandas as pd
import psycopg2
import time

### Extrair

In [ ]:
def get_db_connection():
    while True:
        try:
            conexao = psycopg2.connect(
                host="postgres",
                port=5432,
                database="pokedex_db",
                user="pokedex_user",
                password="pokedex_password"
            )
            return conexao
        except psycopg2.OperationalError:
            print("O banco não está pronto, aguardando 3 segundos...")
            time.sleep(3)

# Get data from database instead of CSV
conexao = get_db_connection()
df = pd.read_sql_query("SELECT * FROM pokemon", conexao)
conexao.close()

print("Dados carregados do banco de dados:")
print(df.head())

### Transformar


In [9]:
# Apaga colunas
colunas_para_apagar = [
'hit_points',
'total_stats',
'base_happiness',
'evolves_from',
'mythical',
'genderless', 
'female_rate', 
'egg_cycles'
]

df_tratado = df.drop(columns=colunas_para_apagar)

# alterando nome do id 
df_tratado = df.rename(columns={'pokedex_number': 'SRK_pok'})


print(df_tratado.head())

print("\n transformação concluída!")

   SRK_pok   pokemon_name type_1  type_2  height  weight  hit_points  attack  \
0        1      Bulbasaur  Grass  Poison     0.7     6.9          45      49   
1        2        Ivysaur  Grass  Poison     1.0    13.0          60      62   
2        3  Mega Venusaur  Grass  Poison     2.4   155.5          80     100   
3        4     Charmander   Fire             0.6     8.5          39      52   
4        5     Charmeleon   Fire             1.1    19.0          58      64   

   defense  total_stats  ...  against_ground  against_flying  against_psychic  \
0       49          318  ...             1.0             2.0              2.0   
1       63          405  ...             1.0             2.0              2.0   
2      123          625  ...             1.0             2.0              2.0   
3       43          309  ...             2.0             1.0              1.0   
4       58          405  ...             2.0             1.0              1.0   

   against_bug against_rock agai

### Carregar 

##### salva dado tratado carregando em um novo csv

In [ ]:
# Load data to database (Gold layer)
conexao = get_db_connection()
cursor = conexao.cursor()

# Create Gold table
cursor.execute("""
CREATE TABLE IF NOT EXISTS pokemon_gold (
    SRK_pok INT NOT NULL PRIMARY KEY,
    pokemon_name VARCHAR(50) NOT NULL,
    type_1 VARCHAR(50) NOT NULL,
    type_2 VARCHAR(50),
    height DOUBLE PRECISION NOT NULL,
    weight DOUBLE PRECISION NOT NULL,
    attack INT NOT NULL,
    defense INT NOT NULL,
    capture_rate INT NOT NULL,
    generation INT NOT NULL,
    base_experience INT NOT NULL,
    exp_type VARCHAR(50) NOT NULL,
    mega_evolution BOOLEAN NOT NULL,
    alolan_form BOOLEAN NOT NULL,
    galarian_form BOOLEAN NOT NULL,
    forms_switchable BOOLEAN NOT NULL,
    legendary BOOLEAN NOT NULL,
    against_normal DOUBLE PRECISION NOT NULL,
    against_fire DOUBLE PRECISION NOT NULL,
    against_water DOUBLE PRECISION NOT NULL,
    against_electric DOUBLE PRECISION NOT NULL,
    against_grass DOUBLE PRECISION NOT NULL,
    against_ice DOUBLE PRECISION NOT NULL,
    against_fighting DOUBLE PRECISION NOT NULL,
    against_poison DOUBLE PRECISION NOT NULL,
    against_ground DOUBLE PRECISION NOT NULL,
    against_flying DOUBLE PRECISION NOT NULL,
    against_psychic DOUBLE PRECISION NOT NULL,
    against_bug DOUBLE PRECISION NOT NULL,
    against_rock DOUBLE PRECISION NOT NULL,
    against_ghost DOUBLE PRECISION NOT NULL,
    against_dragon DOUBLE PRECISION NOT NULL,
    against_dark DOUBLE PRECISION NOT NULL,
    against_steel DOUBLE PRECISION NOT NULL,
    against_fairy DOUBLE PRECISION NOT NULL
)
""")

##### popula dados no banco

In [ ]:
# Insert data into Gold table
for index, row in df_tratado.iterrows():
    cursor.execute("""
    INSERT INTO pokemon_gold (
        SRK_pok, pokemon_name, type_1, type_2, height, weight, 
        attack, defense, capture_rate, generation, base_experience, 
        exp_type, mega_evolution, alolan_form, galarian_form, 
        forms_switchable, legendary, against_normal, against_fire, 
        against_water, against_electric, against_grass, against_ice, 
        against_fighting, against_poison, against_ground, against_flying, 
        against_psychic, against_bug, against_rock, against_ghost, 
        against_dragon, against_dark, against_steel, against_fairy
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, tuple(row))

conexao.commit()

# Fechando conexão
cursor.close()
conexao.close()

print("ETL Gold concluído! Dados carregados na tabela pokemon_gold.")

concluido!
